# Clean nav_history.csv

In [1]:
import pandas as pd

nav = pd.read_csv("D:\\BlueStock FineTech\\data\\raw\\02_nav_history.csv")

# Date conversion
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Sort
nav = nav.sort_values(
    ['amfi_code', 'date']
)

# Remove duplicates
nav = nav.drop_duplicates(
    subset=['amfi_code', 'date']
)

# NAV numeric
nav['nav'] = pd.to_numeric(
    nav['nav'],
    errors='coerce'
)

# Forward fill missing NAV
nav['nav'] = (
    nav.groupby('amfi_code')['nav']
       .ffill()
)

# Validate NAV > 0
invalid_nav = nav[nav['nav'] <= 0]

print("Invalid NAV rows:", len(invalid_nav))

nav = nav[nav['nav'] > 0]

nav.to_csv(
    "data/processed/nav_history_clean.csv",
    index=False
)

Invalid NAV rows: 0


# Clean investor_transactions.csv

In [3]:
tx = pd.read_csv(
    "D:\\BlueStock FineTech\\data\\raw\\08_investor_transactions.csv"
)

tx['transaction_type'] = (
    tx['transaction_type']
      .str.strip()
      .str.title()
)

valid_types = [
    'Sip',
    'Lumpsum',
    'Redemption'
]

invalid_types = tx[
    ~tx['transaction_type'].isin(valid_types)
]

print("Invalid transaction types:",
      len(invalid_types))

Invalid transaction types: 0


In [10]:
tx['transaction_type'].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [5]:
tx.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,Sip,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,Sip,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,Sip,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [9]:
tx['transaction_type'] = (
    tx['transaction_type']
    .replace({'Sip':'SIP'})
)

In [8]:
tx.shape

(32778, 13)

In [13]:
tx.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [ ]:
tx['transaction_date'] = pd.to_datetime(
    tx['transaction_date'],
    errors='coerce'
)

# Validate Amount

In [14]:
invalid_amount = tx[
    tx['amount_inr'] <= 0
]

print(
    "Invalid Amount Rows:",
    len(invalid_amount)
)

tx = tx[
    tx['amount_inr'] > 0
]

Invalid Amount Rows: 0


# KYC Validate

In [15]:
print(
    tx['kyc_status'].unique()
)

['Verified' 'Pending']


In [16]:
valid_kyc = [
    'Verified',
    'Pending',
    'Rejected'
]

invalid_kyc = tx[
    ~tx['kyc_status'].isin(valid_kyc)
]

print(
    "Invalid KYC:",
    len(invalid_kyc)
)

Invalid KYC: 0


In [17]:
tx.to_csv(
    "D:\\BlueStock FineTech\\data\\processed\\investor_transactions_clean.csv",
    index=False
)

# Clean scheme_performance.csv

In [23]:
perf = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct'
]

for col in return_cols:

    perf[col] = pd.to_numeric(
        perf[col],
        errors='coerce'
    )

    print(
        col,
        perf[col].isnull().sum()
    )

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0
benchmark_3yr_pct 0


In [22]:
perf.head(2)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate


# Flag Anamolies

In [24]:
anomalies = perf[
    (perf['return_1yr_pct'] < -100)
    |
    (perf['return_1yr_pct'] > 200)
]

print(
    "Return anomalies:",
    len(anomalies)
)

Return anomalies: 0


# Expense Ratio Check

In [25]:
expense_anomalies = perf[
    (perf['expense_ratio_pct'] < 0.1)
    |
    (perf['expense_ratio_pct'] > 2.5)
]

print(
    "Expense Ratio Issues:",
    len(expense_anomalies)
)

Expense Ratio Issues: 0


In [28]:
perf.to_csv(
    "data/processed/scheme_performance_clean.csv",
    index=False
)

# Schema.SQL

In [34]:
import pandas as pd

files = [
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

for f in files:
    path = f"data/raw/{f}"

    df = pd.read_csv(path)

    print("\n" + "="*60)
    print(f)
    print(df.columns.tolist())
    print(df.head())


03_aum_by_fund_house.csv
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
         date           fund_house  aum_lakh_crore  aum_crore  num_schemes
0  2022-03-31      SBI Mutual Fund            6.05     605000          186
1  2022-03-31  ICICI Prudential MF            4.65     465000          216
2  2022-03-31     HDFC Mutual Fund            4.35     435000          195
3  2022-03-31      Nippon India MF            2.70     270000          177
4  2022-03-31    Kotak Mahindra MF            2.70     270000          168

04_monthly_sip_inflows.csv
['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']
     month  sip_inflow_crore  active_sip_accounts_crore  \
0  2022-01             11517                       4.91   
1  2022-02             11438                       4.93   
2  2022-03             12328                       5.09   
3  2022-04             11863                       5.48   
4  2022-

In [38]:
aum = pd.read_csv("data/raw/03_aum_by_fund_house.csv")

aum['date'] = pd.to_datetime(
    aum['date'],
    errors='coerce'
)

In [39]:
aum.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            90 non-null     datetime64[ns]
 1   fund_house      90 non-null     object        
 2   aum_lakh_crore  90 non-null     float64       
 3   aum_crore       90 non-null     int64         
 4   num_schemes     90 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 3.6+ KB


In [46]:
print(aum.isnull().sum())

date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64


In [47]:
print(
    aum.duplicated().sum()
)

aum = aum.drop_duplicates()

0


In [44]:
numeric_cols = [
    'aum_lakh_crore',
    'aum_crore',
    'num_schemes'
]

for col in numeric_cols:
    print(
        col,
        (aum[col] <= 0).sum()
    )

aum_lakh_crore 0
aum_crore 0
num_schemes 0


In [45]:
aum.to_csv(
    "data/processed/aum_by_fund_house_clean.csv",
    index=False
)

# monthly_sip_inflows.csv Cleaning

In [52]:
sip = pd.read_csv(
    "data/raw/04_monthly_sip_inflows.csv"
)

sip['month'] = pd.to_datetime(
    sip['month'],
    format='%Y-%m'
)

In [61]:
sip.shape

(48, 6)

In [53]:
sip.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   month                      48 non-null     datetime64[ns]
 1   sip_inflow_crore           48 non-null     int64         
 2   active_sip_accounts_crore  48 non-null     float64       
 3   new_sip_accounts_lakh      48 non-null     float64       
 4   sip_aum_lakh_crore         48 non-null     float64       
 5   yoy_growth_pct             36 non-null     float64       
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 2.4 KB


In [60]:
sip.isnull().sum()

month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

# Validate Numeric Values

In [62]:
cols = [
    'sip_inflow_crore',
    'active_sip_accounts_crore',
    'new_sip_accounts_lakh',
    'sip_aum_lakh_crore'
]

for col in cols:
    print(
        col,
        (sip[col] < 0).sum()
    )

sip_inflow_crore 0
active_sip_accounts_crore 0
new_sip_accounts_lakh 0
sip_aum_lakh_crore 0


In [64]:
# Save file
sip.to_csv(
    "data/processed/monthly_sip_inflows_clean.csv",
    index=False
)

# category_inflows.csv Cleaning

In [67]:
cat = pd.read_csv(
    "data/raw/05_category_inflows.csv"
)

cat['month'] = pd.to_datetime(
    cat['month'],
    format='%Y-%m'
)

# Category Standarization

In [69]:
cat['category'] = (
    cat['category']
    .str.strip()
    .str.title()
)
cat = cat.drop_duplicates()

In [72]:
cat.duplicated().sum()

np.int64(0)

In [73]:
cat.to_csv(
    "data/processed/category_inflows_clean.csv",
    index=False
)

# Industry_folio_count.csv Cleaning

In [76]:
folio = pd.read_csv(
    "data/raw/06_industry_folio_count.csv"
)

folio['month'] = pd.to_datetime(
    folio['month'],
    format='%Y-%m'
)

In [79]:
folio.duplicated().sum()

np.int64(0)

# Check Negative Value

In [80]:
cols = [
    'total_folios_crore',
    'equity_folios_crore',
    'debt_folios_crore',
    'hybrid_folios_crore',
    'others_folios_crore'
]

for col in cols:
    print(
        col,
        (folio[col] < 0).sum()
    )

total_folios_crore 0
equity_folios_crore 0
debt_folios_crore 0
hybrid_folios_crore 0
others_folios_crore 0


In [81]:
folio.to_csv(
    "data/processed/industry_folio_count_clean.csv",
    index=False
)

# Portfolio_holdings.csv Cleaning

In [85]:
holdings = pd.read_csv(
    "data/raw/09_portfolio_holdings.csv"
)

holdings['portfolio_date'] = pd.to_datetime(
    holdings['portfolio_date']
)

In [86]:
holdings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   amfi_code          322 non-null    int64         
 1   stock_symbol       322 non-null    object        
 2   stock_name         322 non-null    object        
 3   sector             322 non-null    object        
 4   weight_pct         322 non-null    float64       
 5   market_value_cr    322 non-null    float64       
 6   current_price_inr  322 non-null    float64       
 7   portfolio_date     322 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 20.3+ KB


In [87]:
# Weight validate

invalid_weight = holdings[
    holdings['weight_pct'] <= 0
]

print(
    len(invalid_weight)
)

0


In [88]:
# Price Validate

invalid_price = holdings[
    holdings['current_price_inr'] <= 0
]

print(
    len(invalid_price)
)

0


In [89]:
# Market Value Validation

invalid_mv = holdings[
    holdings['market_value_cr'] <= 0
]

print(
    len(invalid_mv)
)

0


In [93]:
# Duplicate Check

holdings = holdings.drop_duplicates()

,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31
...,...,...,...,...,...,...,...,...
317,149324,DIVISLAB,Divi's Laboratories,Pharma,9.54,855.88,7453.05,2025-12-31
318,149324,BHARTIARTL,Bharti Airtel Ltd,Telecom,1.79,969.71,924.41,2025-12-31
319,149324,TCS,Tata Consultancy Services,IT,13.17,1637.69,1927.89,2025-12-31
320,149324,INDUSINDBK,IndusInd Bank Ltd,Banking,7.63,1387.30,3208.86,2025-12-31


In [95]:
# Save

holdings.to_csv(
    "data/processed/portfolio_holdings_clean.csv",
    index=False
)

# Benchmark_indices.csv Cleaning

In [105]:
benchmark = pd.read_csv(
    "data/raw/10_benchmark_indices.csv"
)

benchmark['date'] = pd.to_datetime(
    benchmark['date']
)

In [108]:
benchmark.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [111]:
benchmark.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         8050 non-null   datetime64[ns]
 1   index_name   8050 non-null   object        
 2   close_value  8050 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 188.8+ KB


In [104]:
benchmark.isnull().sum()

date           0
index_name     0
close_value    0
dtype: int64

In [107]:
# Close Value Validation

invalid_close = benchmark[
    benchmark['close_value'] <= 0
]

print(
    len(invalid_close)
)

0


In [110]:
benchmark = benchmark.drop_duplicates()

In [112]:
# Save

benchmark.to_csv(
    "data/processed/benchmark_indices_clean.csv",
    index=False
)

# Fund_Master.CSV Cleaning

In [114]:
fund_master = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

# Remove duplicates
fund_master = fund_master.drop_duplicates()

# Remove extra spaces
fund_master['fund_house'] = (
    fund_master['fund_house']
    .str.strip()
)

fund_master['category'] = (
    fund_master['category']
    .str.strip()
)

fund_master['sub_category'] = (
    fund_master['sub_category']
    .str.strip()
)

fund_master['risk_category'] = (
    fund_master['risk_category']
    .str.strip()
)

# AMFI code numeric
fund_master['amfi_code'] = pd.to_numeric(
    fund_master['amfi_code'],
    errors='coerce'
)

# Remove rows with missing AMFI code
fund_master = fund_master.dropna(
    subset=['amfi_code']
)

fund_master.to_csv(
    "data/processed/fund_master_clean.csv",
    index=False
)

In [115]:
fund_master.head(3)

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03


In [116]:
import os

files = os.listdir("data/processed")

print(files)
print("\nTotal Files =", len(files))

['aum_by_fund_house_clean.csv', 'benchmark_indices_clean.csv', 'category_inflows_clean.csv', 'fund_master_clean.csv', 'industry_folio_count_clean.csv', 'investor_transactions_clean.csv', 'monthly_sip_inflows_clean.csv', 'nav_history_clean.csv', 'portfolio_holdings_clean.csv', 'scheme_performance_clean.csv']

Total Files = 10


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

query = """
SELECT
    scheme_name,
    aum_crore
FROM scheme_performance
ORDER BY aum_crore DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,scheme_name,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,49046
1,Kotak Emerging Equity Fund - Regular - Growth,47469
2,Nippon India Small Cap Fund - Regular - Growth,43630
3,DSP Top 100 Equity Fund - Regular - Growth,41828
4,UTI Mid Cap Fund - Regular - Growth,41728


In [11]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

query = """
SELECT
    transaction_type,
    ROUND(AVG(amount_inr),2) AS avg_amount
FROM investor_transactions
GROUP BY transaction_type
"""

result = pd.read_sql(query, conn)

print(result)

  transaction_type  avg_amount
0          Lumpsum   254456.02
1       Redemption   250558.79
2              SIP    11018.13
